# 📊 S6E8: Predicting Smartphone Addiction — Comprehensive EDA & Insights
### Exploratory Data Analysis, Synthetic Generator Artifacts, Non-Linear Shapes, & Budget Constraints

Notebook ini didedikasikan untuk melakukan **Exploratory Data Analysis (EDA) mendalam** pada dataset kompetisi **Kaggle Playground Series S6E8**.

### 🎯 Tujuan & Poin Utama yang Dianalisis:
1. **Distribusi Data & Missing Values (MCAR Analysis):** Membuktikan apakah missing value membawa sinyal atau murni acak.
2. **Bentuk Hubungan Non-Linear (Steep S-Curves):** Menganalisis mengapa model linier gagal dan *tree-based models* sangat unggul.
3. **Fenomena Zigzag & Discrete Lattice Artifacts:** Membongkar rahasia fitur `notifications_per_day` & `app_opens_per_day` yang memiliki korelasi linear ~0 tapi bernilai prediktif tinggi.
4. **Time Budget Constraint & Fitur Residual (`other_screen`):** Memvalidasi hukum alokasi waktu layar generator Kaggle.
5. **Anomali Non-Monoton:** Menganalisis mengapa kenaikan screen time non-sosial justru menurunkan risiko kecanduan.
6. **Perilaku Fitur Kategorikal:** Menilai seberapa besar sinyal dari fitur `gender`, `stress_level`, dan `academic_work_impact`.

## 1. Setup Environment & Visualization Styles

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Modern curated color palette
BLUE, ORANGE, GREEN, RED, GRAY = '#2a78d6', '#eb6834', '#1baf7a', '#e34948', '#898781'
SURFACE, GRID = '#fcfcfb', '#e1e0d9'

plt.rcParams.update({
    'figure.facecolor': SURFACE,
    'axes.facecolor': SURFACE,
    'axes.edgecolor': GRID,
    'axes.grid': True,
    'grid.color': GRID,
    'grid.linewidth': 0.8,
    'font.size': 10,
    'figure.dpi': 110
})

# Auto-detect Project Root (works locally & Kaggle)
def get_project_root() -> Path:
    if Path('../data/raw').exists() or Path('../data').exists():
        return Path('..').resolve()
    if Path('data/raw').exists() or Path('data').exists():
        return Path('.').resolve()
    if Path('/kaggle/working').exists():
        return Path('/kaggle/working').resolve()
    return Path('.').resolve()

PROJECT_ROOT = get_project_root()
CANDIDATES = [
    PROJECT_ROOT / 'data' / 'raw',
    PROJECT_ROOT / 'data',
    Path('/kaggle/input/playground-series-s6e8'),
    Path('/kaggle/input/competitions/playground-series-s6e8'),
]
DATA_DIR = next((p for p in CANDIDATES if (p / 'train.csv').exists()), None)

if DATA_DIR is None:
    hits = glob.glob('**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(os.path.dirname(hits[0])).resolve()
    else:
        raise FileNotFoundError('train.csv tidak ditemukan!')

print(f'✅ Project Root : {PROJECT_ROOT}')
print(f'✅ Data Dir     : {DATA_DIR}')

## 2. Dataset Loading & Global Overview

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

TARGET = 'addicted_label'
ID_COL = 'id'

NUM_COLS = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time'
]
CAT_COLS = ['gender', 'stress_level', 'academic_work_impact']
ALL_FEATURES = NUM_COLS + CAT_COLS

print(f'📊 Train Dataset : {train.shape[0]:,} baris, {train.shape[1]} kolom')
print(f'📊 Test Dataset  : {test.shape[0]:,} baris, {test.shape[1]} kolom')

display(train.head())

### Target Variable Distribution (`addicted_label`)

In [ ]:
target_counts = train[TARGET].value_counts().sort_index()
target_rates = target_counts / len(train)
overall_mean = train[TARGET].mean()

fig, ax = plt.subplots(figsize=(8, 2))
left = 0
for label, rate, color, name in zip(target_rates.index, target_rates.values, [GRAY, BLUE], ['0: Not Addicted', '1: Addicted']):
    ax.barh([0], [rate], left=left, height=0.5, color=color, label=f'{name} ({rate:.1%})')
    ax.text(left + rate / 2, 0, f'{rate:.1%}\n({target_counts[label]:,})', ha='center', va='center', color='white', fontweight='bold')
    left += rate

ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_title(f'Target Variable Distribution (Total Train: {len(train):,})', fontsize=12, fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
plt.tight_layout()
plt.show()

print(f'Rasio Positif (Addicted = 1): {overall_mean:.4f} (~70.9%)')
print(f'Rasio Negatif (Addicted = 0): {1 - overall_mean:.4f} (~29.1%)')

### Train vs Test Value Ranges Consistency Check

In [ ]:
range_check = pd.DataFrame({
    'Train Min': train[NUM_COLS].min(),
    'Train Max': train[NUM_COLS].max(),
    'Test Min': test[NUM_COLS].min(),
    'Test Max': test[NUM_COLS].max(),
    'Train Mean': train[NUM_COLS].mean(),
    'Test Mean': test[NUM_COLS].mean()
})
display(range_check.round(2))
print('✅ Rentang nilai fitur numerik antara Train dan Test 100% konsisten (tidak ada pergeseran distribusi nilai ekstrem).')

## 3. Missing Value Analysis & MCAR Statistical Testing

Kita menganalisis apakah data hilang membawa sinyal informatif (*MAR/MNAR*) atau murni terjadi secara acak (*MCAR*).

In [ ]:
# 1. Missing rate comparison Train vs Test
missing_rates = pd.DataFrame({
    'train': train[ALL_FEATURES].isna().mean(),
    'test': test[ALL_FEATURES].isna().mean()
}).sort_values('train')

y_pos = np.arange(len(missing_rates))
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(y_pos + 0.2, missing_rates['train'], height=0.38, color=BLUE, label='Train')
ax.barh(y_pos - 0.2, missing_rates['test'], height=0.38, color=ORANGE, label='Test')
ax.set_yticks(y_pos)
ax.set_yticklabels(missing_rates.index)
ax.set_xlabel('Missing Rate')
ax.set_title('Missing Rate per Feature — Train vs Test')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2. Two-Sample Z-Test for MCAR
z_rows = []
p_overall = train[TARGET].mean()

for c in ALL_FEATURES:
    m = train[c].isna()
    n1, n2 = m.sum(), (~m).sum()
    p1, p2 = train.loc[m, TARGET].mean(), train.loc[~m, TARGET].mean()
    z = (p1 - p2) / np.sqrt(p_overall * (1 - p_overall) * (1 / n1 + 1 / n2))
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    z_rows.append({
        'Feature': c,
        'Missing %': f'{m.mean():.1%}',
        'P(Addicted | Missing)': p1,
        'P(Addicted | Present)': p2,
        'Gap (% pt)': (p1 - p2) * 100,
        'z-stat': z,
        'p-value': p_val
    })

z_df = pd.DataFrame(z_rows).sort_values('p-value')
display(z_df.round(4))
print('📌 Kesimpulan Statistik: Dengan koreksi Bonferroni (alpha = 0.05 / 12 = 0.0042), TIDAK ADA fitur dengan p-value signifikan.')
print('Artinya missing values murni MCAR (Missing Completely At Random) dan penambahan flag `_isna` tidak membawa sinyal.')

## 4. Target Relationships with Numerical Features (The Steep S-Curve)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharey=True)
corrs = train[NUM_COLS].corrwith(train[TARGET])

for ax, col in zip(axes.ravel(), NUM_COLS):
    sub = train[[col, TARGET]].dropna()
    bins = pd.qcut(sub[col], 20, duplicates='drop')
    g = sub.groupby(bins, observed=True)[TARGET].mean()
    centers = [iv.mid for iv in g.index]
    
    ax.axhline(overall_mean, color=GRAY, linestyle='--', lw=1)
    ax.plot(centers, g.values, marker='o', ms=4, color=BLUE, lw=2)
    ax.set_title(f'{col}\n(Pearson r = {corrs[col]:+.3f})', fontsize=9.5, fontweight='bold')
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True)

for ax in axes[:, 0]:
    ax.set_ylabel('P(Addicted)')
    
plt.suptitle('Non-Linear Relationship: Positivity Rate by Feature Value (20 Quantile Bins)', y=1.01, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Discrete Lattice Artifacts & The Zigzag Discovery

Mengapa `notifications_per_day` dan `app_opens_per_day` memiliki korelasi linear mendekati nol, tetapi sangat kuat dalam memprediksi target?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, col in zip(axes, ['notifications_per_day', 'daily_screen_time_hours']):
    g = train.groupby(col, observed=True)[TARGET].agg(['mean', 'size'])
    g = g[g['size'] >= 200]
    
    ax.plot(g.index, g['mean'], color=BLUE, lw=0.8, marker='o', ms=2.5, alpha=0.6, label='Per-value Target Rate')
    ax.plot(g.index, g['mean'].rolling(25, center=True, min_periods=5).median(), color=ORANGE, lw=2.5, label='Rolling Trend (25 pts)')
    ax.axhline(overall_mean, color=GRAY, linestyle='--', lw=1.2)
    ax.set_title(f'{col} (Pearson r = {train[col].corr(train[TARGET]):+.3f})', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend(loc='lower left')
    ax.grid(True)

axes[0].set_ylabel('P(Addicted)')
plt.suptitle('Discrete Zigzag Artifact vs Smooth Trend', y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Temuan: `notifications_per_day` meloncat-loncat tajam antar nilai integer (misal nilai 24 -> 82% addicted, nilai 41 -> 37% addicted).')
print('Ini adalah lookup table buatan generator. Menyetel max_bin tinggi di LightGBM menangkap pola diskrit ini!')

## 6. Time Budget Constraint & The Residual Feature (`other_screen`)

Membuktikan secara empiris aturan generator sintetis:
$$\text{daily\_screen\_time\_hours} \ge \text{social\_media\_hours} + \text{gaming\_hours} + \text{work\_study\_hours}$$

In [ ]:
parts = train[['daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours']].dropna()
parts_sum = parts['social_media_hours'] + parts['gaming_hours'] + parts['work_study_hours']
residual = parts['daily_screen_time_hours'] - parts_sum

print('--- Bukti Budget Constraint ---')
print(f'Total baris lengkap               : {len(parts):,}')
print(f'Jumlah pelanggaran (residual < 0) : {(residual < -1e-9).sum()} (0.00%)')
print(f'Rata-rata sisa waktu (residual)   : {residual.mean():.3f} jam')
print(f'Median sisa waktu (residual)      : {residual.median():.3f} jam')

# Visualisasi Scatter Constraint
fig, ax = plt.subplots(figsize=(7, 6))
sample_pts = parts.sample(15000, random_state=42)
sample_sum = sample_pts['social_media_hours'] + sample_pts['gaming_hours'] + sample_pts['work_study_hours']

ax.scatter(sample_sum, sample_pts['daily_screen_time_hours'], s=8, alpha=0.2, color=BLUE, label='Data Points')
ax.plot([0, 14], [0, 14], color=ORANGE, linestyle='--', lw=2.5, label='Batas Minimal (Daily = Sum of Parts)')
ax.set_xlabel('Social + Gaming + Work/Study (Hours)')
ax.set_ylabel('Daily Screen Time (Hours)')
ax.set_title('Batas Waktu: Daily Screen Time Tidak Pernah Kurang Dari Jumlah Komponennya', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### Univariate Feature Predictive Power (AUC Leaderboard)

In [ ]:
auc_scores = []
for col in NUM_COLS:
    sub = train[[col, TARGET]].dropna()
    score = roc_auc_score(sub[TARGET], sub[col])
    if score < 0.5:
        score = 1 - score
    auc_scores.append({'Feature': col, 'Single Feature AUC': score, 'Type': 'Raw Feature'})

# Hitung AUC untuk fitur rekayasa other_screen
sub_res = pd.DataFrame({'other_screen': residual, TARGET: parts[TARGET]})
auc_res = roc_auc_score(sub_res[TARGET], sub_res['other_screen'])
auc_scores.append({'Feature': 'other_screen (Residual)', 'Single Feature AUC': auc_res, 'Type': 'Engineered Residual'})

auc_table = pd.DataFrame(auc_scores).sort_values('Single Feature AUC', ascending=False)
display(auc_table.round(4))
print('🚀 Fitur rekayasa `other_screen` menempati posisi ke-4 dari seluruh fitur, membuktikan kekuatan besar fitur residual ini!')

## 7. The Non-Monotone Anomaly (Mengapa Monotone Constraint Gagal)

In [ ]:
g = train[['daily_screen_time_hours', 'social_media_hours', TARGET]].dropna()

# Filter baris dengan social media rendah (1.0 hingga 1.5 jam)
sl = g[(g['social_media_hours'] > 1.0) & (g['social_media_hours'] <= 1.5)]
dip_rows = []
for lo in range(3, 10):
    c = sl[(sl['daily_screen_time_hours'] > lo) & (sl['daily_screen_time_hours'] <= lo + 1)]
    if len(c) > 200:
        dip_rows.append({
            'Daily Screen Time': f'{lo}-{lo+1}h',
            'P(Addicted)': c[TARGET].mean(),
            'Sample Size': len(c)
        })

dip_df = pd.DataFrame(dip_rows)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dip_df['Daily Screen Time'], dip_df['P(Addicted)'], marker='o', color=RED, lw=2.5, label='Social Media in (1.0h, 1.5h]')
ax.axhline(overall_mean, color=GRAY, linestyle='--', label=f'Overall Mean ({overall_mean:.1%})')
ax.set_xlabel('Daily Screen Time (Hours)')
ax.set_ylabel('P(Addicted)')
ax.set_title('Anomali: Menahan Social Media Tetap -> Menambah Screen Time Menurunkan P(Addicted)!', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

display(dip_df.round(3))
print('📌 Mengapa terjadi penurunan? Karena ketika waktu sosial media dibatasi konstan, screen time tambahan dialokasikan ke belajar/gaming yang kurang memicu adiksi.')

## 8. Categorical Features Analysis (`gender`, `stress_level`, `academic_work_impact`)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)

for ax, col in zip(axes, CAT_COLS):
    sub = train.groupby(col, observed=True)[TARGET].agg(['mean', 'size']).sort_index()
    ax.bar(sub.index.astype(str), sub['mean'], color=BLUE, alpha=0.8, width=0.5)
    ax.axhline(overall_mean, color=GRAY, linestyle='--', lw=1.2)
    for i, (rate, size) in enumerate(zip(sub['mean'], sub['size'])):
        ax.text(i, rate + 0.015, f'{rate:.3f}', ha='center', fontweight='bold', fontsize=9)
        ax.text(i, 0.05, f'n={size:,}', ha='center', color='white', fontsize=8)
    ax.set_title(col, fontweight='bold')
    ax.set_ylim(0, 0.9)
    ax.grid(True)

axes[0].set_ylabel('P(Addicted)')
plt.suptitle('Target Positivity Rate by Categorical Features (All Nearly Flat)', y=1.02, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Fitur kategorikal memiliki selisih positivity rate yang sangat kecil (maksimal 2.2% pt di gender), sehingga tidak perlu target encoding rumit pada kategori.')

## 9. Key Actionable Takeaways for Modeling

| No | Aspek Data | Temuan EDA | Aksi Rekomendasi Modeling |
|:---|:---|:---|:---|
| 1 | **Missingness** | MCAR (Missing Completely At Random) | Jangan imputasi mean/median buatan; biarkan GBDT menangani secara *native*. |
| 2 | **Budget Constraint** | $\text{daily} \ge \text{social} + \text{gaming} + \text{work}$ (100% konsisten) | Buat fitur residual `other_screen = daily - sum(parts)` (AUC ~0.76). |
| 3 | **Discrete Lattice** | `notifications` & `app_opens` berupa *zigzag lookup table* | Set `max_bin=512~2047` di LightGBM & gunakan *Factorization Machines*. |
| 4 | **Monotonicity** | Terjadi penurunan kurva pada penambahan screen time non-sosial | Jangan gunakan *monotone constraints*. |
| 5 | **Validation** | 691k baris dengan perbandingan 71:29 | Gunakan 5-Fold Stratified K-Fold (`seed=42`) untuk OOF & Stacking. |